# 04b — Conduct and efficiencies: why do merger predictions miss?

Notebook 04 compares simulated and realised Alaska–Hawaiian fare changes. A simulation can miss because demand is
wrong, because the merged firm's costs changed (efficiencies or integration costs), or because conduct is not
what was assumed (the course notes' conjectural-variations lecture and the Week 7 warning that "assuming the wrong
conduct can lead to misspecification"). This notebook separates the last two using observed post-merger prices.

**1. Market-wide conduct before the merger.** Let the ownership matrix put weight $\kappa$ on rivals' profits
($\kappa=0$ Bertrand–Nash, $\kappa=1$ joint profit maximisation). For each $\kappa$, implied marginal costs
$c(\kappa)=p-\Delta(\kappa)^{-1}s$ are regressed on cost shifters and fixed effects; under the true conduct the
cost residual is uncorrelated with markup shifters excluded from costs (the BLP rival-characteristic instruments).
The GMM objective over a grid of $\kappa$ shows which conduct the data support (Berry and Haile 2014;
Backus, Conlon and Sinkinson 2021).

**2. Conduct versus efficiencies after the merger.** With pre-merger demand held fixed, observed post-merger prices
imply post-merger marginal costs for every assumed degree $\kappa_{AH}$ of joint pricing between AS and HA. The
difference-in-differences change in implied log marginal cost (AS/HA overlap products vs western controls)
traces a curve: full joint pricing ($\kappa_{AH}=1$) implies large cost savings if prices rose little, while no
coordination ($\kappa_{AH}=0$) implies no savings. Without outside cost information the data identify only this
curve, which is the honest summary (compare Miller and Weinberg 2017; Aryal, Chattopadhyaya and Ciliberto 2025).

In [ ]:
%run 00_setup.ipynb

In [ ]:
%run 00_model_functions.ipynb

In [ ]:
PARTIES = ("AS", "HA")
IV = IV_SETS["baseline"]
WEST = {"WA", "OR", "CA", "NV", "AZ", "ID", "UT", "MT", "WY", "CO", "NM", "AK", "HI"}
BASE = {qlabel(*q) for q in [(2023, 4), (2024, 1), (2024, 2)]}
POSTQ = {qlabel(*q) for q in AS_HA_POST}
p_all = pd.read_parquet(INTERIM / "products_airport_nondir_iv.parquet")
val = load_json("asha_validation")
PRE = val["pre_params"]
MODELS = {"NL (air travel)": ("nested_air", PRE["nested_air"]),
          "RC demand+supply": ("rc", {"sigma": PRE["rc"]["sigma"], "pi": PRE["rc"]["pi"]})}

## 1. Market-wide conduct, pre-merger quarters

In [ ]:
def ownership_kappa(car, kappa, pairs=None):
    """Ownership matrix: 1 within a carrier; kappa across carriers (all pairs, or only the listed pair)."""
    O = (car[:, None] == car[None, :]).astype(float)
    if pairs is None:
        O[O == 0] = kappa
    else:
        a, b = np.isin(car, pairs[0]), np.isin(car, pairs[1])
        O[np.ix_(a, b)] = kappa; O[np.ix_(b, a)] = kappa
    return O

def market_jacobians(res, p):
    """Demand Jacobians dS_j/dp_k by market, computed once and reused for every conduct assumption."""
    Jm = res.compute_demand_jacobians()
    return {m: Jm[idx][:, :len(idx)] for m, idx in p.groupby("market_ids").indices.items()}

def implied_costs(jac, p, kappa, pairs=None, markets=None):
    """c = p + (O o J')^{-1} s from the first-order conditions s + (O o J')(p - c) = 0."""
    c = np.full(len(p), np.nan)
    pos = p.groupby("market_ids").indices
    for m in (markets if markets is not None else pos):
        idx = pos[m]
        O = ownership_kappa(p.carrier.values[idx], kappa, pairs)
        c[idx] = p.prices.values[idx] + np.linalg.solve(O * jac[m].T, p.shares.values[idx])
    return c

rng = np.random.default_rng(3)
pre = p_all[p_all.t <= qindex(2024, 2)]
cond_routes = rng.choice(pre.route.unique(), size=200 if SMOKE else 1500, replace=False)
cs_sample = pre[pre.route.isin(cond_routes)].sort_values(["market_ids", "carrier", "nonstop"]).reset_index(drop=True)
X3 = ["cs_fuel", "cs_pilot", "cs_exfuel", "nonstop", "distance", "hub_presence"]

def conduct_objective(c, d, ivs):
    """Two-step GMM J statistic of the cost equation ln c = X3 g + FE + w with instruments X3 + markup shifters."""
    ok = c > 1e-3
    d = d[ok]
    y = np.log(c[ok])
    D = pd.get_dummies(d[["quarter", "carrier_fe"]].astype(str), drop_first=True).astype(float)
    V = pd.concat([pd.Series(y, index=d.index, name="y"), d[X3 + ivs], D], axis=1)
    R = demean_multi(V, [d.route])
    cl = d.route.astype("category").cat.codes
    exog = [c_ for c_ in X3 + list(D.columns) if R[c_].std() > 1e-8]
    # IVGMM with no endogenous regressors: overidentification of E[Z w] = 0 through the instruments
    from linearmodels.iv import IVGMM
    m = IVGMM(R.y, R[exog[1:]], R[[exog[0]]], R[[exog[0]] + ivs], weight_type="clustered", clusters=cl) \
        .fit(cov_type="clustered", clusters=cl)
    return float(m.j_stat.stat), float((~ok).mean())

grid = [0.0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]
cond_rows = []
for name, (kind, prm) in MODELS.items():
    with pyblp.parallel(N_CORES):
        r = evaluate_fixed(cs_sample, kind, prm, IV)
    jac = market_jacobians(r, cs_sample)
    assert np.allclose(implied_costs(jac, cs_sample, 0.0), r.compute_costs()[:, 0], atol=1e-6)
    for k in grid:
        c = implied_costs(jac, cs_sample, k)
        J, neg = conduct_objective(c, cs_sample, BLP_IVS + COUNT_IVS)
        cond_rows.append({"model": name, "kappa": k, "J": J, "share mc<=0": neg,
                          "median Lerner": float(np.nanmedian((cs_sample.prices - c) / cs_sample.prices))})
        print(name, k, round(J, 1), round(neg, 3))
cond = pd.DataFrame(cond_rows)
save_table(cond.set_index(["model", "kappa"]), "conduct_test", float_format="%.3f",
           note="Cost-equation GMM objective (Hansen J) for implied marginal costs under conduct $\\kappa$ (weight on rivals' profits; "
                "0 = Bertrand--Nash, 1 = joint profit maximisation). Instruments: cost shifters plus BLP rival characteristics and "
                f"counts. Random sample of {len(cond_routes)} routes, 2022Q1--2024Q2.")
fig, ax = plt.subplots(figsize=(5.2, 3.2))
for i, (name, d) in enumerate(cond.groupby("model", sort=False)):
    ax.plot(d.kappa, d.J, marker="o", label=name, color=PALETTE[i])
ax.set_xlabel("Conduct $\\kappa$ (weight on rivals' profits)"); ax.set_ylabel("GMM objective (J)"); ax.legend()
fig.tight_layout(); savefig(fig, "conduct_objective")
cond

## 2. Alaska–Hawaiian: implied post-merger costs under alternative joint-pricing assumptions

In [ ]:
apt_state = pd.read_parquet(CLEAN / "airport_cbsa.parquet").set_index("AIRPORT").AIRPORT_STATE_CODE
d = p_all[p_all.quarter.isin(BASE | POSTQ)].copy()
party_routes = set(p_all.loc[p_all.carrier.isin(PARTIES), "route"])
base_both = d[d.quarter.isin(BASE)].groupby("market_ids").carrier.agg(lambda s: set(PARTIES) <= set(s))
overlap_routes = {m.split("|")[0] for m in base_both.index[base_both]}
west = d.a.map(apt_state).isin(WEST) | d.b.map(apt_state).isin(WEST)
d = d[d.route.isin(overlap_routes) | ((~d.route.isin(party_routes)) & west)]
if SMOKE:
    d = d[d.route.isin(list(overlap_routes)[:15] + list(d.loc[~d.route.isin(overlap_routes), "route"].unique()[:60]))]
d = d.sort_values(["market_ids", "carrier", "nonstop"]).reset_index(drop=True)
d["post"] = d.quarter.isin(POSTQ).astype(int)
d["treated"] = d.route.isin(overlap_routes) & d.carrier.isin(PARTIES)
d["product"] = d.route + ":" + d.carrier + ":" + d.nonstop.astype(str)
print(f"{d.route.isin(overlap_routes).sum():,} overlap-route products, {(~d.route.isin(overlap_routes)).sum():,} control products")

def did_logmc(c):
    x = d.assign(lmc=np.log(np.clip(c, 1e-3, None)))
    x = x[(x.route.isin(overlap_routes) & x.carrier.isin(PARTIES)) | (~x.route.isin(overlap_routes))]
    x["tp"] = (x.treated & (x.post == 1)).astype(float)
    ab = pd.DataFrame({"product": x["product"].astype("category"), "quarter": x.quarter.astype("category")})
    f = AbsorbingLS(x.lmc, x[["tp"]], absorb=ab, weights=x.passengers).fit(cov_type="clustered", clusters=x.route.astype("category").cat.codes)
    return float(f.params["tp"]), float(f.std_errors["tp"])

curve_rows = []
pos = d.groupby("market_ids").indices
post_both = [m for m, idx in pos.items() if d.post.values[idx[0]] == 1 and set(PARTIES) <= set(d.carrier.values[idx])]
for name, (kind, prm) in MODELS.items():
    with pyblp.parallel(N_CORES):
        r = evaluate_fixed(d, kind, prm, IV)
    c_base = r.compute_costs()[:, 0]                  # Bertrand with separate AS and HA (pre-merger conduct)
    jac = market_jacobians(r, d)
    for k in [0.0, 0.25, 0.5, 0.75, 1.0]:
        c = c_base.copy()
        if k > 0:
            ck = implied_costs(jac, d, k, pairs=(["AS"], ["HA"]), markets=post_both)
            sel = ~np.isnan(ck)
            c[sel] = ck[sel]
        b, se = did_logmc(c)
        curve_rows.append({"model": name, "kappa AS-HA (post)": k, "DiD log mc": b, "s.e.": se,
                           "implied mc change (%)": 100 * (np.exp(b) - 1)})
        print(name, k, round(100 * (np.exp(b) - 1), 2))
curve = pd.DataFrame(curve_rows)
save_table(curve.set_index(["model", "kappa AS-HA (post)"]), "asha_conduct_efficiency", float_format="%.3f",
           note="Difference-in-differences change in implied log marginal cost of AS/HA products on overlap routes relative to "
                "western control products (2023Q4--2024Q2 vs 2024Q4--2025Q2), when post-merger prices are rationalised with weight "
                "$\\kappa$ on the partner's profits. Negative values are implied efficiencies; pre-merger demand held fixed.")

fig, ax = plt.subplots(figsize=(5.2, 3.2))
for i, (name, g) in enumerate(curve.groupby("model", sort=False)):
    ax.errorbar(g["kappa AS-HA (post)"], g["implied mc change (%)"], yerr=196 * g["s.e."], marker="o", capsize=2, label=name, color=PALETTE[i])
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Assumed joint pricing between AS and HA after the merger ($\\kappa$)")
ax.set_ylabel("Implied change in marginal cost (%)"); ax.legend(fontsize=8)
fig.tight_layout(); savefig(fig, "asha_conduct_efficiency")
curve

In [ ]:
if True:
    save_json({"conduct": cond.to_dict("records"), "curve": curve.to_dict("records")}, "conduct_efficiency")